# Setwise distillation lab

The teacher is an LLM that compares **k documents at once** and picks the most relevant one. The student scores **one document at a time**. Training uses a **setwise loss** so that, inside each set, the teacher's winner gets the highest probability. Inference sorts by the student's score. There is no LLM call and no loss at inference.

Three pieces are left open on purpose:

1. `SetwiseTeacher.compare_documents_setwise` in `src/models/teacher.py`
2. `setwise_loss` in `src/loss/setwise.py`
3. `StudentRanker` in `src/models/student.py` (`__init__` and `score_pairs`)

The prompt you already validated is in `notebooks/jusbrasil_soc_llms_setwise.ipynb`. Heapsort, label files, the training loop, and nDCG are already wired.

Human `relevance` is only for nDCG. The training target is the teacher winner.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import DistillConfig
from src.data.schema import load_examples, save_examples
from src.models.teacher import SetwiseTeacher

## Data

Same columns as the setwise pilot: `query`, `title`, `body`, `court`, `relevance`.

In [ ]:
cfg = DistillConfig(
    dataset_name="jusbrasil",
    teacher_client="ollama",
    teacher_model="llama3.1:8b",
    k_size=4,
    sets_per_query=2,
    max_doc_chars=1500,
)

DATA_DIR = Path("/Users/david/Documents/data/jusbrasil/manual/with_body/splits")
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "vali.parquet")
train_df.head(2)

## 1. Teacher

Implement `compare_documents_setwise`. It should return the index of the winning document, or `-1` if the reply is not a valid letter.

`format_doc` is already shared with evaluation, so the student later reads the same text the teacher judged. `rank` is the setwise heapsort and starts working as soon as `compare_documents_setwise` does.

In [ ]:
teacher = SetwiseTeacher(
    api_client_type=cfg.teacher_client,
    model_name=cfg.teacher_model,
    k_size=cfg.k_size,
    dataset_name=cfg.dataset_name,
    max_chars=cfg.max_doc_chars,
)

sample = train_df.groupby("query").head(cfg.k_size)
query_text = sample["query"].iloc[0]
docs = sample.to_dict(orient="records")
print(teacher.format_doc(docs[0])[:400])
print("winner index:", teacher.compare_documents_setwise(docs, query_text))

## 2. Labels

`from_samples` draws random sets of size `k`. Start with `max_queries` small so a mistake in the parser does not spend a full epoch of calls.

`from_rank_trace` is the alternative: run the heapsort and keep every comparison it made. Use that when you want the student to imitate the teacher's full ranking procedure rather than random subsets.

In [ ]:
from src.engine.generator import SetwiseLabelGenerator

generator = SetwiseLabelGenerator(
    teacher,
    sets_per_query=cfg.sets_per_query,
    seed=cfg.seed,
)
generated = generator.from_samples(train_df, max_queries=2)
print(len(generated.examples), "examples,", generated.skipped, "skipped")
generated.examples[0]

In [ ]:
LABELS_PATH = ROOT / "notebooks" / "data" / "setwise_labels.jsonl"
save_examples(generated.examples, LABELS_PATH)
examples = load_examples(LABELS_PATH)
len(examples)

## 3. Setwise loss

Implement `setwise_loss`. For each set, the student scores are logits and the teacher winner is the class. Average the cross-entropy over the batch.

Do not use this function at inference. Inference sorts by the raw score.

`listmle_loss` and `pairwise_ranknet_loss` are optional. Leave them until the setwise path trains.

In [ ]:
import torch

from src.loss.setwise import setwise_loss

scores = torch.tensor([[2.0, 0.0, 0.0]])
winner = torch.tensor([0])
value = float(setwise_loss(scores, winner))
print(value)
assert abs(value - 0.2395) < 1e-3

## 4. Student

Implement a pointwise scorer in `StudentRanker`. `score_sets` is already written: it flattens each set, calls `score_pairs`, and reshapes to `[batch, k]`.

A cross-encoder is the usual model. Load it in `__init__` and return one logit per pair from `score_pairs`. Higher means more relevant. Install Transformers with `poetry add transformers` when you get there.

The trainer raises a clear error if the module has no parameters.

In [ ]:
from src.engine.trainer import SetwiseLoader, SetwiseTrainer
from src.models.student import StudentRanker

student = StudentRanker(cfg.student_model)
loader = SetwiseLoader(examples, batch_size=cfg.batch_size, shuffle=True, seed=cfg.seed)
trainer = SetwiseTrainer(student, setwise_loss, lr=cfg.learning_rate)
history = trainer.fit(loader, epochs=cfg.epochs)
history

## 5. Pointwise inference

Each candidate is scored alone with `teacher.format_doc`, then sorted. nDCG uses the human grade, in the same definition as the setwise pilot.

In [ ]:
from src.engine.evaluator import evaluate_pointwise

metrics = evaluate_pointwise(student, val_df, format_fn=teacher.format_doc)
metrics.means